In [0]:

storage_account_name = "flightstacc"
storage_account_key = dbutils.secrets.get(scope = 'wikimedia', key = 'Access-key')

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/wikipedia/batch_historical/2026/*/*.json"

df = spark.read.option("multiline", "true").json(path)
display(df)

In [0]:

from pyspark.sql.functions import col, explode_outer

df_exploded = df.select(
    col("items")
)

display(df_exploded)

In [0]:
from pyspark.sql.functions import explode, col

df_step1 = df.withColumn("item", explode(col("items")))

df_step2 = df_step1.withColumn("article_data", explode(col("item.articles")))

df_final = df_step2.withColumn("project", col("item.project")) \
                   .withColumn("access", col("item.access")) \
                   .withColumn("year", col("item.year")) \
                   .withColumn("month", col("item.month")) \
                   .withColumn("day", col("item.day")) \
                   .withColumn("article", col("article_data.article")) \
                   .withColumn("views", col("article_data.views")) \
                   .withColumn("rank", col("article_data.rank")) \
                   .drop("items", "item", "article_data") 

display(df_final)

In [0]:

output_stream_path = "abfss://bronze@flightstacc.dfs.core.windows.net/snowflake_staging_batch/batch_data/"
df_final.write.mode("overwrite").parquet(output_stream_path)